# Day 1: Workflow Analysis

## Module 5: Automation in AI | Al Jazira Bank

In this notebook you will explore the AJB workflow inventory dataset, examine process characteristics, and begin identifying automation candidates based on data.

### What you will do
1. Load and inspect the workflow inventory dataset.
2. Explore distributions of manual hours, error rates, and complexity.
3. Identify patterns that distinguish strong automation candidates from poor ones.
4. Prepare observations to support your Lab A and Lab B deliverables.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# Load the workflow inventory
workflows = pd.read_csv("../data/workflow_inventory.csv")

print(f"Dataset shape: {workflows.shape}")
print(f"Columns: {list(workflows.columns)}")
workflows.head()

In [ ]:
# Basic statistics for numeric columns
print("Summary statistics:")
print(workflows[["steps_count", "manual_hours_weekly", "error_rate_pct"]].describe())

print("\nAutomation potential distribution:")
print(workflows["automation_potential"].value_counts())

print("\nComplexity distribution:")
print(workflows["complexity"].value_counts())

print("\nDepartment distribution:")
print(workflows["department"].value_counts())

In [ ]:
# Exercise: Explore the relationship between manual hours and error rate
#
# Task 1: Create a scatter plot of manual_hours_weekly vs error_rate_pct.
# Task 2: Colour or label the points by automation_potential.
# Task 3: Which workflows have both high hours AND high error rates?
#          These are your strongest automation candidates.

fig, ax = plt.subplots(figsize=(10, 6))

colours = {"High": "#2ecc71", "Medium": "#f39c12", "Low": "#e74c3c"}
for potential, group in workflows.groupby("automation_potential"):
    ax.scatter(
        group["manual_hours_weekly"],
        group["error_rate_pct"],
        label=potential,
        c=colours.get(potential, "grey"),
        s=80,
        alpha=0.8,
    )
    for _, row in group.iterrows():
        ax.annotate(
            row["workflow_id"],
            (row["manual_hours_weekly"], row["error_rate_pct"]),
            fontsize=7,
            ha="left",
            va="bottom",
        )

ax.set_xlabel("Manual Hours per Week")
ax.set_ylabel("Error Rate (%)")
ax.set_title("Workflow Inventory: Manual Hours vs Error Rate")
ax.legend(title="Automation Potential")
plt.tight_layout()
plt.show()

In [ ]:
# Exercise: Identify your top candidates
#
# Sort workflows by a composite of manual_hours_weekly and error_rate_pct.
# Filter for High or Medium automation potential.
# Write your observations below.

candidates = workflows[workflows["automation_potential"].isin(["High", "Medium"])].copy()
candidates["priority_score"] = (
    candidates["manual_hours_weekly"] / candidates["manual_hours_weekly"].max() * 0.5
    + candidates["error_rate_pct"] / candidates["error_rate_pct"].max() * 0.5
)
candidates = candidates.sort_values("priority_score", ascending=False)

print("Top candidates by composite score (hours + error rate):")
print(
    candidates[
        ["workflow_id", "process_name", "department", "manual_hours_weekly",
         "error_rate_pct", "automation_potential", "complexity", "priority_score"]
    ].head(10).to_string(index=False)
)

# Reflection:
# - Which workflows appear in your top 5?
# - Are any high-scoring workflows also high complexity? What does that mean for implementation?
# - Would you change the weighting between hours and error rate? Why?